In [11]:
import pandas as pd
import numpy as np
import pickle as pkl
import umap.umap_ as umap
import plotly.express as px
import os

In [12]:
df = pd.read_csv('/workspaces/dev_container_template/data/depth_all_holes.csv')
print(df.shape)
df.head()

(81886, 4)


,file_name,hole_id,depth_from,depth_to
0,SU-671-W1_0.0_0.5.jpg,SU-671-W1,0.0,0.5
1,SU-671-W1_0.5_1.0.jpg,SU-671-W1,0.5,1.0
2,SU-671-W1_1.0_1.5.jpg,SU-671-W1,1.0,1.5
3,SU-671-W1_1.5_2.0.jpg,SU-671-W1,1.5,2.0
4,SU-671-W1_2.0_2.5.jpg,SU-671-W1,2.0,2.5


In [13]:
with open('/workspaces/dev_container_template/data/features.pkl', 'rb') as f:
    feats = pkl.load(f)
print(feats.shape)
feats

/tmp/ipykernel_3052/3777060219.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  feats = pkl.load(f)


(81886, 2)


,file_name,feature
0,SU-671-W1_0.0_0.5.jpg,"[1.2882447, 2.1303527, 2.6275353, 1.7057508, 1..."
1,SU-671-W1_0.5_1.0.jpg,"[1.0163176, 4.6862416, 1.16729, 1.6992068, 1.4..."
2,SU-671-W1_1.0_1.5.jpg,"[1.198943, 5.587632, -0.014981463, 0.51072556,..."
3,SU-671-W1_1.5_2.0.jpg,"[1.2575666, 1.5112426, 1.0605989, 1.7607124, 1..."
4,SU-671-W1_10.0_10.5.jpg,"[1.4044752, 5.0177755, 3.058706, 1.2419477, 0...."
...,...,...
81881,VU-4534_97.5_98.0.jpg,"[0.98364073, 2.1615348, 3.204988, 3.3533967, 0..."
81882,VU-4534_98.0_98.5.jpg,"[0.14159597, 2.5276785, 3.1975365, 1.3573023, ..."
81883,VU-4534_98.5_99.0.jpg,"[0.57093346, 7.3649797, 0.9798129, 0.16416411,..."
81884,VU-4534_99.0_99.5.jpg,"[2.5856795, 0.4808417, 2.0127451, 0.56760633, ..."


In [17]:
type(feats)

pandas.DataFrame

In [18]:
df = df[df['hole_id'] == 'SU-671-W1']

feats = feats[feats['file_name'].str.contains('SU-671-W1', na=False)]
df.shape, feats.shape

((302, 4), (302, 2))

In [19]:
# Merge on filename
df = df.merge(feats, on='file_name', how='outer')
print(df.shape)
df.head()

(302, 5)


,file_name,hole_id,depth_from,depth_to,feature
0,SU-671-W1_0.0_0.5.jpg,SU-671-W1,0.0,0.5,"[1.2882447, 2.1303527, 2.6275353, 1.7057508, 1..."
1,SU-671-W1_0.5_1.0.jpg,SU-671-W1,0.5,1.0,"[1.0163176, 4.6862416, 1.16729, 1.6992068, 1.4..."
2,SU-671-W1_1.0_1.5.jpg,SU-671-W1,1.0,1.5,"[1.198943, 5.587632, -0.014981463, 0.51072556,..."
3,SU-671-W1_1.5_2.0.jpg,SU-671-W1,1.5,2.0,"[1.2575666, 1.5112426, 1.0605989, 1.7607124, 1..."
4,SU-671-W1_10.0_10.5.jpg,SU-671-W1,10.0,10.5,"[1.4044752, 5.0177755, 3.058706, 1.2419477, 0...."


In [20]:
# Features are numpy arrays of length 256
print(df['feature'][0].shape)
display(df['feature'][0][:10])

(3584,)


array([ 1.2882447e+00,  2.1303527e+00,  2.6275353e+00,  1.7057508e+00,
        1.8637087e+00,  7.9349232e+00,  3.7785694e-01, -4.2069776e-05,
        6.0784960e-01,  2.4082518e+00], dtype=float32)

In [21]:
# Convert to a single column each
for i in range(256):
    df[f'feat_{i}'] = df['feature'].apply(lambda x: x[i])

# Drop the original feature column
df = df.drop(columns=['feature'])

df.head()

/tmp/ipykernel_3052/2383683615.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'feat_{i}'] = df['feature'].apply(lambda x: x[i])


,file_name,hole_id,depth_from,depth_to,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,...,feat_246,feat_247,feat_248,feat_249,feat_250,feat_251,feat_252,feat_253,feat_254,feat_255
0,SU-671-W1_0.0_0.5.jpg,SU-671-W1,0.0,0.5,1.288245,2.130353,2.627535,1.705751,1.863709,7.934923,...,1.533833,1.069707,8.379168,4.019193,1.048077,0.985321,-0.009970,0.250723,1.260098,4.600509
1,SU-671-W1_0.5_1.0.jpg,SU-671-W1,0.5,1.0,1.016318,4.686242,1.167290,1.699207,1.462381,2.541518,...,1.228821,1.744789,0.681160,2.354458,0.742294,0.877970,0.977536,1.689950,5.273170,-0.022270
2,SU-671-W1_1.0_1.5.jpg,SU-671-W1,1.0,1.5,1.198943,5.587632,-0.014981,0.510726,3.816632,1.725811,...,1.197059,-0.035124,3.590303,0.685262,3.862692,0.733118,9.345230,4.580661,1.069667,-0.002861
3,SU-671-W1_1.5_2.0.jpg,SU-671-W1,1.5,2.0,1.257567,1.511243,1.060599,1.760712,1.076801,1.229465,...,4.146112,0.041945,0.641651,-0.009805,1.189716,1.455460,0.349330,-0.061770,3.840922,0.302050
4,SU-671-W1_10.0_10.5.jpg,SU-671-W1,10.0,10.5,1.404475,5.017776,3.058706,1.241948,0.495051,2.846248,...,2.087965,0.045532,8.943372,4.775162,3.607162,-0.003565,4.541287,3.990798,2.573992,3.211162


# Unsupervised clustering

In [22]:
# We just want the numerical (data) columns
feat_cols = [c for c in df.columns if 'feat' in c]
data = df[feat_cols].values

# Train the umap reducer
reducer = umap.UMAP(
    n_neighbors = 100,   # neighbours on manifold
    min_dist = 0.2,       # tightness of embedding - min distance points are allowed to be apart
    n_components = 3,   # number of arbitrary dimensions to project to
    metric = 'cosine',  # how distance is computed on the manifold
    n_jobs = 8,
    #random_state = 13,  # set constant to make it perfectly reproducible
)
embedding = reducer.fit_transform(data)

# merge with original data
new_colnames = {i:f'umap_{i}' for i in range(embedding.shape[1])}
df_umap = pd.DataFrame(embedding).rename(columns=new_colnames).merge(df.reset_index(drop = True), left_index = True, right_index = True).copy()

In [28]:
df_umap.to_csv('/workspaces/dev_container_template/data/umap_embedding.csv', index=False)

In [23]:
# plot in 3d
fig = px.scatter_3d(
    df_umap,
    x = 'umap_0',
    y = 'umap_1',
    z = 'umap_2',
    # color = 'GeoRckInterpType',
    #color_discrete_map=colour_map, 
)
fig.update_traces(marker=dict(size=5, line=dict(width=0.5,color='black')))

# Customize the layout
fig.update_layout(
    title='UMAP of sample image features', # Title
    width=1600,  # Set the width of the figure to 800 pixels
    height=1200,  # Set the height of the figure to 600 pixels
    plot_bgcolor='rgba(250, 250, 250, 1)',
)

In [50]:
colour_map = {
    'ZWA':'#edf8b1', 
    'ZWTT':'#41b6c4', 
    'ZWT':'#1d91c0', 
    'ZPF':'#225ea8', 
    'DGR':'#bd0026', 
    'MDY':'#fc4e2a', 
    'BXV':'#006d2c', 
    'HMBx':'#238b45', 
    'BXG':'#74c476',
    'FVO':'grey', 
    'SHZ':'grey', 
    'FLT':'grey', 
    'HEMQV':'#41ab5d', 
    'XLC':'white', 
    'ZWP':'#c7e9b4', 
    'GRV':'#feb24c', 
    'ZRS':'#253494', 
    'ZWAW':'#ffffd9',
    'ZWC':'#7fcdbb'
}

In [51]:
# plot in 3d
fig = px.scatter_3d(
    df_3h_umap,
    x = 'umap_0',
    y = 'umap_1',
    z = 'umap_2',
    color = 'GeoRck',
    color_discrete_map=colour_map, 
)
fig.update_traces(marker=dict(size=5, line=dict(width=0.5,color='black')))

# Customize the layout
fig.update_layout(
    title='UMAP of sample image features', # Title
    width=1600,  # Set the width of the figure to 800 pixels
    height=1200,  # Set the height of the figure to 600 pixels
    plot_bgcolor='rgba(250, 250, 250, 1)',
)

In [25]:
# plot in 3d
fig = px.scatter_3d(
    df_umap[df_umap['GeoRckInterpType']=='Basement'],
    x = 'umap_0',
    y = 'umap_1',
    z = 'umap_2',
    color = 'photo_gen',
    #color_discrete_map=colour_map, 
)
fig.update_traces(marker=dict(size=5, line=dict(width=0.5,color='black')))

# Customize the layout
fig.update_layout(
    title='UMAP of sample image features', # Title
    width=1600,  # Set the width of the figure to 800 pixels
    height=1200,  # Set the height of the figure to 600 pixels
    plot_bgcolor='rgba(250, 250, 250, 1)',
)

In [31]:
df_umap.groupby('GeoRckInterp')['photo_gen'].value_counts()

GeoRckInterp  photo_gen
BXG           2023         1835
              2025          619
BXV           2025           39
              2023           14
DGR           2023         1432
              2025         1034
DOL           2023          162
              2025           27
FVO           2023            9
GRV           2025            5
              2023            3
HEMQ          2023          817
              2025          456
HMBx          2025          963
              2023          515
MDY           2025           72
              2023           29
SDBx          2025           83
              2023           12
SHZ           2023            3
VEIN          2023            1
              2025            1
XLC           2023          583
              2025          140
ZPF           2023         2147
              2025         2119
ZRS           2025            9
              2023            3
ZWA           2025          262
              2023           72
ZWC           20

In [39]:
# plot in 3d
fig = px.scatter_3d(
    df_umap[df_umap['GeoRckInterp']=='BXG'],
    x = 'umap_0',
    y = 'umap_1',
    z = 'umap_2',
    color = 'hole_id',
    #color_discrete_map=colour_map, 
)
fig.update_traces(marker=dict(size=5, line=dict(width=0.5,color='black')))

# Customize the layout
fig.update_layout(
    title='UMAP of image features from BXG', # Title
    width=1600,  # Set the width of the figure to 800 pixels
    height=1200,  # Set the height of the figure to 600 pixels
    plot_bgcolor='rgba(250, 250, 250, 1)',
)

In [40]:
# plot in 3d
fig = px.scatter_3d(
    df_umap[df_umap['GeoRckInterp']=='BXG'],
    x = 'umap_0',
    y = 'umap_1',
    z = 'umap_2',
    color = 'photo_gen',
    #color_discrete_map=colour_map, 
)
fig.update_traces(marker=dict(size=5, line=dict(width=0.5,color='black')))

# Customize the layout
fig.update_layout(
    title='UMAP of image features from BXG', # Title
    width=1600,  # Set the width of the figure to 800 pixels
    height=1200,  # Set the height of the figure to 600 pixels
    plot_bgcolor='rgba(250, 250, 250, 1)',
)

In [ ]:
# plot in 3d
fig = px.scatter_3d(
    df_umap[df_umap['GeoRckInterp']=='ZPF'],
    x = 'umap_0',
    y = 'umap_1',
    z = 'umap_2',
    color = 'hole_id',
    #color_discrete_map=colour_map, 
)
fig.update_traces(marker=dict(size=5, line=dict(width=0.5,color='black')))

# Customize the layout
fig.update_layout(
    title='UMAP of sample image features from ZPF', # Title
    width=1600,  # Set the width of the figure to 800 pixels
    height=1200,  # Set the height of the figure to 600 pixels
    plot_bgcolor='rgba(250, 250, 250, 1)',
)

In [41]:
# plot in 3d
fig = px.scatter_3d(
    df_umap[df_umap['GeoRckInterp']=='ZPF'],
    x = 'umap_0',
    y = 'umap_1',
    z = 'umap_2',
    color = 'photo_gen',
    #color_discrete_map=colour_map, 
)
fig.update_traces(marker=dict(size=5, line=dict(width=0.5,color='black')))

# Customize the layout
fig.update_layout(
    title='UMAP of sample image features from ZPF', # Title
    width=1600,  # Set the width of the figure to 800 pixels
    height=1200,  # Set the height of the figure to 600 pixels
    plot_bgcolor='rgba(250, 250, 250, 1)',
)